# Phase 7 Diagnosis + Fix

**Goal:** Understand why Phase 7 autoregressive accuracy is 66% on 8-bit,
then fix it.

**Plan (run cells in order):**
1. Setup + model + data (cells 1-3)
2. Load Phase 7 checkpoint (cell 4)
3. **Diagnose** — find where errors occur (cell 5)
4. **Constrained decoding** — restrict invalid tokens at inference (cell 6)
5. **Eval constrained decoding** (cell 7)
6. **DAgger without BACKTRACK** — if still needed (cell 8)
7. Final eval + save (cells 9-10)


In [ ]:
# Cell 1: Setup
import os
if not os.path.exists('/content/recursive-transformers/src'):
    !git clone https://github.com/dhruvsyam123/recursive-transformers.git /content/recursive-transformers
else:
    !cd /content/recursive-transformers && git pull
%cd /content/recursive-transformers
!pip install -q jax[cuda12] equinox optax jaxtyping numpy pyyaml matplotlib einops

import jax
print(f'JAX devices: {jax.devices()}')
print(f'JAX backend: {jax.default_backend()}')


In [ ]:
# Cell 2: Model — same as Phase 5 but with PAUSE/RESUME tokens
import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import math
import numpy as np

from src.model.looped_transformer import (
    MultiHeadSelfAttention, FeedForward, RMSNorm, count_parameters,
)
from src.data.karatsuba_trace import int_to_bits, bits_to_int

# --- Extended vocabulary (inline, no source file changes) ---
# Original token IDs from src/data/tokenizer.py
PAD_ID = 0
SEP_ID = 1
INPUT_ID = 2
SPLIT_ID = 3
SUB_MUL_0_ID = 4
SUB_MUL_1_ID = 5
SUB_MUL_2_ID = 6
ADD_ID = 7
SUB_ID = 8
COMBINE_ID = 9
OUTPUT_ID = 10
BASE_MUL_ID = 11
BIT_0_ID = 12
BIT_1_ID = 13
# New tokens (appended at end of vocab to preserve all existing IDs)
PAUSE_ID = 143
RESUME_ID = 144
VOCAB_SIZE = 145

# Step types for position encoding
STEP_INPUT = 0
STEP_SPLIT = 1
STEP_SUB_MUL_0 = 2
STEP_SUB_MUL_1 = 3
STEP_SUB_MUL_2 = 4
STEP_ADD = 5
STEP_SUB = 6
STEP_COMBINE = 7
STEP_OUTPUT = 8
STEP_BASE_MUL = 9
STEP_PAUSE = 10
STEP_RESUME = 11

TAG_TO_STEP = {
    INPUT_ID: STEP_INPUT, SPLIT_ID: STEP_SPLIT,
    SUB_MUL_0_ID: STEP_SUB_MUL_0, SUB_MUL_1_ID: STEP_SUB_MUL_1,
    SUB_MUL_2_ID: STEP_SUB_MUL_2, ADD_ID: STEP_ADD,
    SUB_ID: STEP_SUB, COMBINE_ID: STEP_COMBINE,
    OUTPUT_ID: STEP_OUTPUT, BASE_MUL_ID: STEP_BASE_MUL,
    PAUSE_ID: STEP_PAUSE, RESUME_ID: STEP_RESUME,
}

ALL_TAG_IDS = set(TAG_TO_STEP.keys())

def bit_to_token(b):
    return BIT_0_ID if b == 0 else BIT_1_ID

def token_to_bit(t):
    return 0 if t == BIT_0_ID else 1


# --- Sinusoidal encoding (same as Phase 5) ---
def sinusoidal_encoding(positions, d_model):
    pos = positions.astype(jnp.float32)
    if pos.ndim == 0:
        pos = pos[None]
        squeeze = True
    else:
        squeeze = False
    dim_indices = jnp.arange(0, d_model, 2, dtype=jnp.float32)
    freqs = jnp.exp(-dim_indices * (math.log(10000.0) / d_model))
    angles = pos[:, None] * freqs[None, :]
    enc = jnp.zeros((pos.shape[0], d_model))
    enc = enc.at[:, 0::2].set(jnp.sin(angles))
    enc = enc.at[:, 1::2].set(jnp.cos(angles))
    if squeeze:
        return enc[0]
    return enc


class AllSinusoidalHierarchicalPos(eqx.Module):
    d_model: int = eqx.field(static=True)
    comp_dim: int = eqx.field(static=True)
    def __init__(self, d_model):
        assert d_model % 4 == 0
        self.d_model = d_model
        self.comp_dim = d_model // 4
    def __call__(self, bit_sig, depth, sub_id, step_type):
        return jnp.concatenate([
            sinusoidal_encoding(bit_sig, self.comp_dim),
            sinusoidal_encoding(depth, self.comp_dim),
            sinusoidal_encoding(sub_id, self.comp_dim),
            sinusoidal_encoding(step_type, self.comp_dim),
        ], axis=-1)


class LoopBlock(eqx.Module):
    attention: MultiHeadSelfAttention
    ffn: FeedForward
    norm1: RMSNorm
    norm2: RMSNorm
    d_model: int = eqx.field(static=True)
    def __init__(self, d_model, n_heads, d_ff, *, key):
        k1, k2 = jax.random.split(key)
        self.attention = MultiHeadSelfAttention(d_model, n_heads, key=k1)
        self.ffn = FeedForward(d_model, d_ff, key=k2)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.d_model = d_model
    def __call__(self, x, timestep, mask=None):
        t_emb = sinusoidal_encoding(timestep, self.d_model)
        x_cond = x + t_emb[None, :]
        normed = jax.vmap(self.norm1)(x_cond)
        x = x + self.attention(normed, mask=mask)
        normed = jax.vmap(self.norm2)(x)
        x = x + jax.vmap(self.ffn)(normed)
        return x


class KaratsubaModel(eqx.Module):
    token_embed: eqx.nn.Embedding
    hier_pos: AllSinusoidalHierarchicalPos
    block: LoopBlock
    final_norm: RMSNorm
    output_head: eqx.nn.Linear
    inject_scale: jnp.ndarray
    def __call__(self, tokens, n_loops, bit_sig, depth, sub_id, step_type):
        x0 = jax.vmap(self.token_embed)(tokens)
        pos = self.hier_pos(bit_sig, depth, sub_id, step_type)
        x0 = x0 + pos
        x = x0
        seq_len = tokens.shape[0]
        mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=jnp.bool_))
        def scan_fn(hidden, timestep):
            hidden = hidden + self.inject_scale * x0
            hidden = self.block(hidden, timestep, mask=mask)
            return hidden, None
        x, _ = jax.lax.scan(scan_fn, x, jnp.arange(n_loops))
        x = jax.vmap(self.final_norm)(x)
        return jax.vmap(self.output_head)(x)


D_MODEL = 256
key = jax.random.PRNGKey(42)
k1, k2, k3 = jax.random.split(key, 3)

model = KaratsubaModel(
    token_embed=eqx.nn.Embedding(VOCAB_SIZE, D_MODEL, key=k1),
    hier_pos=AllSinusoidalHierarchicalPos(D_MODEL),
    block=LoopBlock(D_MODEL, n_heads=8, d_ff=512, key=k2),
    final_norm=RMSNorm(D_MODEL),
    output_head=eqx.nn.Linear(D_MODEL, VOCAB_SIZE, key=k3),
    inject_scale=jnp.array(0.1),
)
print(f'Parameters: {count_parameters(model):,}')

In [ ]:
# Cell 3: PAUSE/RESUME trace generator + dataset + mask

# --- Single-level trace generator ---
def generate_single_level_trace(x, y, n_bits, base_case_bits=4):
    """Generate a single-level Karatsuba trace with PAUSE/RESUME.
    Returns list of (token_id, position_dict) tuples.
    
    Base case (n_bits <= base_case_bits): [INPUT] bits [BASE_MUL] product [OUTPUT] product
    Recursive case: [INPUT] bits [SPLIT] ... [SUB_MUL_0] operands [PAUSE] [RESUME] result ... [OUTPUT] result
    """
    assert x >= 0 and y >= 0
    assert x < (1 << n_bits) and y < (1 << n_bits)
    product = x * y
    product_bits = 2 * n_bits
    
    seq = []  # list of (token_id, {bit_significance, step_type})
    
    def add_tag(tag_id, step_type):
        seq.append((tag_id, {'bit_significance': 200 + step_type, 'step_type': step_type}))
    
    def add_bits(bits_list, step_type, bit_offset=0):
        for i, b in enumerate(bits_list):
            seq.append((bit_to_token(b), {'bit_significance': bit_offset + i, 'step_type': step_type}))
    
    # --- Base case ---
    if n_bits <= base_case_bits:
        add_tag(INPUT_ID, STEP_INPUT)
        add_bits(int_to_bits(x, n_bits), STEP_INPUT)
        add_bits(int_to_bits(y, n_bits), STEP_INPUT)
        add_tag(BASE_MUL_ID, STEP_BASE_MUL)
        add_bits(int_to_bits(product, product_bits), STEP_BASE_MUL)
        add_tag(OUTPUT_ID, STEP_OUTPUT)
        add_bits(int_to_bits(product, product_bits), STEP_OUTPUT)
        return seq, product
    
    # --- Recursive case ---
    half = n_bits // 2
    hi_bits = n_bits - half
    mask_low = (1 << half) - 1
    
    x_lo, x_hi = x & mask_low, x >> half
    y_lo, y_hi = y & mask_low, y >> half
    
    # Compute sub-products classically (for ground truth in RESUME)
    z0 = x_lo * y_lo
    z2 = x_hi * y_hi
    sum_x = x_lo + x_hi
    sum_y = y_lo + y_hi
    z1_raw = sum_x * sum_y
    z1 = z1_raw - z0 - z2
    result = z0 + (z1 << half) + (z2 << (2 * half))
    assert result == product, f'{x}*{y}: {result} != {product}'
    
    # Sub-problem bit widths
    z0_n = half
    z2_n = hi_bits
    sum_actual_bits = max(half, hi_bits) + 1
    z1_n = sum_actual_bits
    
    # [INPUT]
    add_tag(INPUT_ID, STEP_INPUT)
    add_bits(int_to_bits(x, n_bits), STEP_INPUT)
    add_bits(int_to_bits(y, n_bits), STEP_INPUT)
    
    # [SPLIT]
    add_tag(SPLIT_ID, STEP_SPLIT)
    add_bits(int_to_bits(x_hi, hi_bits), STEP_SPLIT)
    add_bits(int_to_bits(x_lo, half), STEP_SPLIT)
    add_bits(int_to_bits(y_hi, hi_bits), STEP_SPLIT)
    add_bits(int_to_bits(y_lo, half), STEP_SPLIT)
    
    # [SUB_MUL_0] operands [PAUSE] / [RESUME] z0
    add_tag(SUB_MUL_0_ID, STEP_SUB_MUL_0)
    add_bits(int_to_bits(x_lo, z0_n), STEP_SUB_MUL_0)
    add_bits(int_to_bits(y_lo, z0_n), STEP_SUB_MUL_0)
    add_tag(PAUSE_ID, STEP_PAUSE)
    add_tag(RESUME_ID, STEP_RESUME)
    add_bits(int_to_bits(z0, 2 * z0_n), STEP_RESUME)
    
    # [SUB_MUL_2] operands [PAUSE] / [RESUME] z2
    add_tag(SUB_MUL_2_ID, STEP_SUB_MUL_2)
    add_bits(int_to_bits(x_hi, z2_n), STEP_SUB_MUL_2)
    add_bits(int_to_bits(y_hi, z2_n), STEP_SUB_MUL_2)
    add_tag(PAUSE_ID, STEP_PAUSE)
    add_tag(RESUME_ID, STEP_RESUME)
    add_bits(int_to_bits(z2, 2 * z2_n), STEP_RESUME)
    
    # [ADD] sums
    add_tag(ADD_ID, STEP_ADD)
    add_bits(int_to_bits(sum_x, sum_actual_bits), STEP_ADD)
    add_bits(int_to_bits(sum_y, sum_actual_bits), STEP_ADD)
    
    # [SUB_MUL_1] operands [PAUSE] / [RESUME] z1_raw
    add_tag(SUB_MUL_1_ID, STEP_SUB_MUL_1)
    add_bits(int_to_bits(sum_x, z1_n), STEP_SUB_MUL_1)
    add_bits(int_to_bits(sum_y, z1_n), STEP_SUB_MUL_1)
    add_tag(PAUSE_ID, STEP_PAUSE)
    add_tag(RESUME_ID, STEP_RESUME)
    add_bits(int_to_bits(z1_raw, 2 * z1_n), STEP_RESUME)
    
    # [SUB] z1 = z1_raw - z0 - z2
    add_tag(SUB_ID, STEP_SUB)
    add_bits(int_to_bits(z1, product_bits), STEP_SUB)
    
    # [COMBINE] result
    add_tag(COMBINE_ID, STEP_COMBINE)
    add_bits(int_to_bits(result, product_bits), STEP_COMBINE)
    
    # [OUTPUT] result
    add_tag(OUTPUT_ID, STEP_OUTPUT)
    add_bits(int_to_bits(result, product_bits), STEP_OUTPUT)
    
    return seq, product


# --- Test trace generator ---
print('=== Trace Generator Test ===')
for x, y, nb in [(7, 5, 4), (179, 214, 8), (15, 15, 4), (0, 200, 8)]:
    seq, prod = generate_single_level_trace(x, y, nb)
    expected = x * y
    status = 'OK' if prod == expected else 'FAIL'
    tag_names = {INPUT_ID:'IN', SPLIT_ID:'SP', SUB_MUL_0_ID:'M0', SUB_MUL_1_ID:'M1',
                 SUB_MUL_2_ID:'M2', ADD_ID:'AD', SUB_ID:'SU', COMBINE_ID:'CO',
                 OUTPUT_ID:'OU', BASE_MUL_ID:'BM', PAUSE_ID:'PA', RESUME_ID:'RE'}
    tags = [tag_names.get(t, '.') for t, _ in seq if t in ALL_TAG_IDS]
    print(f'{x:3d} * {y:3d} ({nb}-bit) = {prod} [{status}] {len(seq)} tokens  tags: {"-".join(tags)}')


# --- Dataset creation ---
def make_dataset(bit_widths_and_counts, base_case_bits=4, seed=42):
    """Generate training examples as padded numpy arrays."""
    rng = np.random.RandomState(seed)
    examples = []  # list of (token_ids, positions, product)
    
    for n_bits, n_samples in bit_widths_and_counts:
        max_val = 1 << n_bits
        if n_bits <= 4:  # exhaustive
            pairs = [(x, y) for x in range(max_val) for y in range(max_val)]
        else:
            pairs = [(rng.randint(0, max_val), rng.randint(0, max_val)) for _ in range(n_samples)]
        
        for x, y in pairs:
            seq, prod = generate_single_level_trace(x, y, n_bits, base_case_bits)
            token_ids = np.array([t for t, _ in seq], dtype=np.int32)
            # Positions: just bit_significance and step_type (depth=0, sub_id=0 always)
            bit_sigs = np.array([p['bit_significance'] for _, p in seq], dtype=np.int32)
            step_types = np.array([p['step_type'] for _, p in seq], dtype=np.int32)
            examples.append((token_ids, bit_sigs, step_types, x, y, n_bits, prod))
    
    return examples


def get_batches(examples, batch_size, max_len, shuffle=True, rng_seed=None):
    """Yield padded batches."""
    rng = np.random.RandomState(rng_seed)
    indices = np.arange(len(examples))
    if shuffle:
        rng.shuffle(indices)
    
    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start+batch_size]
        bs = len(batch_idx)
        
        tk = np.zeros((bs, max_len), dtype=np.int32)  # PAD_ID = 0
        bs_arr = np.zeros((bs, max_len), dtype=np.int32)
        st_arr = np.zeros((bs, max_len), dtype=np.int32)
        pm = np.zeros((bs, max_len), dtype=np.float32)
        
        for i, idx in enumerate(batch_idx):
            token_ids, bit_sigs, step_types, x, y, nb, prod = examples[idx]
            sl = len(token_ids)
            tk[i, :sl] = token_ids
            bs_arr[i, :sl] = bit_sigs
            st_arr[i, :sl] = step_types
            pm[i, :sl] = 1.0
        
        yield {
            'token_ids': tk,
            'bit_sig': bs_arr,
            'step_type': st_arr,
            'pad_mask': pm,
        }


# --- Loss mask ---
def make_pause_resume_mask(token_ids_full, pad_mask_full):
    """Mask out INPUT bits and RESUME bits (unpredictable).
    Uses cumulative scan to track most-recent tag.
    Returns mask aligned with targets (shifted by 1)."""
    # Identify tag positions
    is_input_tag = (token_ids_full == INPUT_ID)
    is_resume_tag = (token_ids_full == RESUME_ID)
    is_bit = (token_ids_full == BIT_0_ID) | (token_ids_full == BIT_1_ID)
    
    # Track "after INPUT" and "after RESUME" regions
    # A region ends when any non-bit token (tag) appears
    is_any_tag = jnp.zeros_like(token_ids_full, dtype=jnp.bool_)
    for tid in [INPUT_ID, SPLIT_ID, SUB_MUL_0_ID, SUB_MUL_1_ID, SUB_MUL_2_ID,
                ADD_ID, SUB_ID, COMBINE_ID, OUTPUT_ID, BASE_MUL_ID, PAUSE_ID, RESUME_ID]:
        is_any_tag = is_any_tag | (token_ids_full == tid)
    
    # For each position, what was the last tag? Use cummax trick:
    # Assign each tag position an increasing index, forward-fill for bit positions
    # Then check if that tag was INPUT or RESUME
    #
    # Simpler approach: track cumulative sums of INPUT and RESUME tags,
    # and cumulative sums of ALL tags. A bit is "after INPUT/RESUME" if
    # the count of INPUT+RESUME tags > count of OTHER tags since the last tag.
    #
    # Even simpler: a bit is unpredictable if no non-(INPUT|RESUME) tag has
    # appeared since the last (INPUT|RESUME) tag.
    
    # Forward-fill the last tag ID
    # Use the fact that tag_ids are > 0 and bit tokens we set to 0
    tag_or_zero = jnp.where(is_any_tag, token_ids_full, 0)
    
    # Cumulative max of tag positions (assigns each position its last tag)
    # Since we want forward-fill, we use a scan
    def scan_fill(carry, x):
        val = jnp.where(x > 0, x, carry)
        return val, val
    
    # Vectorize across batch
    def forward_fill_row(row):
        _, filled = jax.lax.scan(scan_fill, jnp.int32(0), row)
        return filled
    
    last_tag = jax.vmap(forward_fill_row)(tag_or_zero)
    
    # A bit is unpredictable if its last_tag is INPUT or RESUME
    unpredictable = is_bit & ((last_tag == INPUT_ID) | (last_tag == RESUME_ID))
    predictable = ~unpredictable
    full_mask = pad_mask_full * predictable.astype(jnp.float32)
    
    # Shift by 1 to align with targets
    return full_mask[:, 1:]


# --- Build datasets ---
print('\n=== Building Datasets ===')
train_4bit = make_dataset([(4, 256)], base_case_bits=4, seed=42)
train_8bit = make_dataset([(8, 4000)], base_case_bits=4, seed=42)

ml4 = max(len(ex[0]) for ex in train_4bit) + 2
ml8 = max(len(ex[0]) for ex in train_8bit) + 2

print(f'4-bit: {len(train_4bit)} examples, max_len={ml4}')
print(f'8-bit: {len(train_8bit)} examples, max_len={ml8}')

# Verify mask on a sample
print('\n=== Mask Verification ===')
for batch in get_batches(train_8bit[:4], batch_size=4, max_len=ml8, shuffle=False):
    tk = jnp.array(batch['token_ids'])
    pm = jnp.array(batch['pad_mask'])
    mask = make_pause_resume_mask(tk, pm)
    
    # Show first example
    tag_names = {INPUT_ID:'[IN]', SPLIT_ID:'[SP]', SUB_MUL_0_ID:'[M0]', SUB_MUL_1_ID:'[M1]',
                 SUB_MUL_2_ID:'[M2]', ADD_ID:'[AD]', SUB_ID:'[SU]', COMBINE_ID:'[CO]',
                 OUTPUT_ID:'[OU]', BASE_MUL_ID:'[BM]', PAUSE_ID:'[PA]', RESUME_ID:'[RE]',
                 BIT_0_ID:'0', BIT_1_ID:'1', PAD_ID:'_'}
    tgt = tk[0, 1:]  # targets
    m = mask[0]
    print(f'{'Pos':>4} {'Input':>6} {'Target':>6} {'Mask':>5}')
    print('-' * 25)
    for i in range(min(30, int(pm[0].sum()) - 1)):
        inp_name = tag_names.get(int(tk[0, i]), '?')
        tgt_name = tag_names.get(int(tgt[i]), '?')
        print(f'{i:4d} {inp_name:>6} {tgt_name:>6} {float(m[i]):>5.0f}')
    break

In [ ]:
# Cell 4: Train Phase 7 model (PAUSE/RESUME, 500 epochs)
# Then save checkpoint so you don't have to retrain next time
import random
import time

schedule = optax.warmup_cosine_decay_schedule(
    init_value=1e-5, peak_value=1e-3, warmup_steps=500,
    decay_steps=50000, end_value=1e-6
)
optimizer = optax.adamw(learning_rate=schedule, weight_decay=0.15)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

def make_train_step(n_loops):
    @eqx.filter_jit
    def train_step(model, opt_state, token_ids, pad_mask, bit_sig, step_type):
        input_ids = token_ids[:, :-1]
        target_ids = token_ids[:, 1:]
        inp_bs = bit_sig[:, :-1]; inp_st = step_type[:, :-1]
        mask = make_pause_resume_mask(token_ids, pad_mask)
        zeros = jnp.zeros_like(inp_bs)
        def loss_fn(model):
            def fwd(ids, bs, st):
                return model(ids, n_loops, bs, zeros[0], zeros[0], st)
            logits = jax.vmap(fwd)(input_ids, inp_bs, inp_st)
            log_probs = jax.nn.log_softmax(logits, axis=-1)
            targets_oh = jax.nn.one_hot(target_ids, VOCAB_SIZE)
            tok_loss = -jnp.sum(targets_oh * log_probs, axis=-1)
            return jnp.sum(tok_loss * mask) / jnp.maximum(jnp.sum(mask), 1.0)
        loss, grads = eqx.filter_value_and_grad(loss_fn)(model)
        updates, new_opt = optimizer.update(grads, opt_state, eqx.filter(model, eqx.is_array))
        return eqx.apply_updates(model, updates), new_opt, loss
    return train_step

train_steps = {n: make_train_step(n) for n in [4, 6, 8]}

# Teacher-forced eval
eval_8bit = make_dataset([(8, 200)], seed=999)
eml8 = max(len(ex[0]) for ex in eval_8bit) + 2

def tf_eval(model, examples, max_len, n_loops=8):
    correct, total = 0, 0
    for batch in get_batches(examples, batch_size=16, max_len=max_len, shuffle=False):
        tk = jnp.array(batch['token_ids']); pm = jnp.array(batch['pad_mask'])
        bs = jnp.array(batch['bit_sig']); st = jnp.array(batch['step_type'])
        inp = tk[:, :-1]; tgt = tk[:, 1:]
        inp_bs = bs[:, :-1]; inp_st = st[:, :-1]
        zeros = jnp.zeros_like(inp_bs)
        em = make_pause_resume_mask(tk, pm)
        def fwd(ids, b, s): return model(ids, n_loops, b, zeros[0], zeros[0], s)
        logits = jax.vmap(fwd)(inp, inp_bs, inp_st)
        preds = jnp.argmax(logits, axis=-1)
        matches = (preds == tgt) | (em == 0)
        correct += int(jnp.all(matches, axis=-1).sum())
        total += tk.shape[0]
    return correct, total

rng = random.Random(42)
t0 = time.time()
print('=== Training Phase 7 model (500 epochs) ===')
for epoch in range(500):
    epoch_loss, n_batches = 0.0, 0
    n_loops = rng.choice([4, 6, 8])
    step_fn = train_steps[n_loops]
    for batch in get_batches(train_4bit, batch_size=64, max_len=ml4):
        tk = jnp.array(batch['token_ids']); pm = jnp.array(batch['pad_mask'])
        bs = jnp.array(batch['bit_sig']); st = jnp.array(batch['step_type'])
        model, opt_state, loss = step_fn(model, opt_state, tk, pm, bs, st)
        epoch_loss += float(loss); n_batches += 1
    n_loops = rng.choice([4, 6, 8])
    step_fn = train_steps[n_loops]
    for batch in get_batches(train_8bit, batch_size=16, max_len=ml8):
        tk = jnp.array(batch['token_ids']); pm = jnp.array(batch['pad_mask'])
        bs = jnp.array(batch['bit_sig']); st = jnp.array(batch['step_type'])
        model, opt_state, loss = step_fn(model, opt_state, tk, pm, bs, st)
        epoch_loss += float(loss); n_batches += 1
    if epoch % 50 == 0:
        avg = epoch_loss / n_batches
        c8, t8 = tf_eval(model, eval_8bit, eml8)
        elapsed = time.time() - t0
        print(f'Epoch {epoch:4d} | Loss: {avg:.4f} | 8-bit TF: {c8}/{t8} = {c8/t8:.1%} | {elapsed:.0f}s')

c8, t8 = tf_eval(model, eval_8bit, eml8)
elapsed = time.time() - t0
print(f'\nTraining complete in {elapsed:.0f}s.')
print(f'Final 8-bit TF: {c8}/{t8} = {c8/t8:.1%}')

# Save checkpoint
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
CKPT_DIR = '/content/drive/MyDrive/karatsuba_checkpoints/'
os.makedirs(CKPT_DIR, exist_ok=True)
eqx.tree_serialise_leaves(os.path.join(CKPT_DIR, 'model_phase7_pause_resume.eqx'), model)
print(f'Checkpoint saved to {CKPT_DIR}')


In [ ]:
# Cell 5: DIAGNOSE — Where do autoregressive errors occur?
import random as pyrandom
import time

# --- JIT-optimized prediction ---
MAX_INFER_LEN = 512

@eqx.filter_jit
def _jit_forward(model, tokens, bit_sigs, step_types):
    zeros = jnp.zeros_like(bit_sigs)
    return model(tokens, 8, bit_sigs, zeros, zeros, step_types)

_inf_tk = np.zeros(MAX_INFER_LEN, dtype=np.int32)
_inf_bs = np.zeros(MAX_INFER_LEN, dtype=np.int32)
_inf_st = np.zeros(MAX_INFER_LEN, dtype=np.int32)

def model_predict_next(model, token_ids, bit_sigs, step_types, n_loops=8):
    seq_len = len(token_ids)
    if seq_len > MAX_INFER_LEN:
        tokens = jnp.array(token_ids, dtype=jnp.int32)
        bs = jnp.array(bit_sigs, dtype=jnp.int32)
        st = jnp.array(step_types, dtype=jnp.int32)
        zeros = jnp.zeros_like(bs)
        logits = model(tokens, n_loops, bs, zeros, zeros, st)
        return int(jnp.argmax(logits[-1]))
    _inf_tk[:seq_len] = token_ids; _inf_tk[seq_len:] = 0
    _inf_bs[:seq_len] = bit_sigs; _inf_bs[seq_len:] = 0
    _inf_st[:seq_len] = step_types; _inf_st[seq_len:] = 0
    logits = _jit_forward(model, jnp.array(_inf_tk), jnp.array(_inf_bs), jnp.array(_inf_st))
    return int(jnp.argmax(logits[seq_len - 1]))


def model_predict_next_with_logits(model, token_ids, bit_sigs, step_types, n_loops=8):
    """Same as above but also returns full logit vector (for confidence analysis)."""
    seq_len = len(token_ids)
    _inf_tk[:seq_len] = token_ids; _inf_tk[seq_len:] = 0
    _inf_bs[:seq_len] = bit_sigs; _inf_bs[seq_len:] = 0
    _inf_st[:seq_len] = step_types; _inf_st[seq_len:] = 0
    logits = _jit_forward(model, jnp.array(_inf_tk), jnp.array(_inf_bs), jnp.array(_inf_st))
    row = logits[seq_len - 1]
    return int(jnp.argmax(row)), row


# --- Inference helpers ---
def extract_sub_problem(token_ids):
    sub_mul_ids = {SUB_MUL_0_ID, SUB_MUL_1_ID, SUB_MUL_2_ID}
    last_sub_pos = -1
    for i in range(len(token_ids) - 1, -1, -1):
        if token_ids[i] in sub_mul_ids:
            last_sub_pos = i; break
    if last_sub_pos == -1: return None
    bits = []
    for i in range(last_sub_pos + 1, len(token_ids)):
        t = token_ids[i]
        if t == BIT_0_ID: bits.append(0)
        elif t == BIT_1_ID: bits.append(1)
        elif t in ALL_TAG_IDS: break
    if len(bits) == 0 or len(bits) % 2 != 0: return None
    half = len(bits) // 2
    return bits_to_int(bits[:half]), bits_to_int(bits[half:]), half


TAG_NAMES = {
    INPUT_ID:'INPUT', SPLIT_ID:'SPLIT', SUB_MUL_0_ID:'M0', SUB_MUL_1_ID:'M1',
    SUB_MUL_2_ID:'M2', ADD_ID:'ADD', SUB_ID:'SUB', COMBINE_ID:'COMBINE',
    OUTPUT_ID:'OUTPUT', BASE_MUL_ID:'BASE_MUL', PAUSE_ID:'PAUSE', RESUME_ID:'RESUME',
    BIT_0_ID:'0', BIT_1_ID:'1', PAD_ID:'PAD',
}
def tok_name(t): return TAG_NAMES.get(t, f'?{t}')


def diagnostic_inference(model, x, y, n_bits, base_case_bits=4, n_loops=8, depth=0):
    """Autoregressive inference that records every model decision for comparison."""
    x_bits = int_to_bits(x, n_bits); y_bits = int_to_bits(y, n_bits)
    token_ids = [INPUT_ID]
    bit_sigs = [200 + STEP_INPUT]
    step_types = [STEP_INPUT]
    for i, b in enumerate(x_bits):
        token_ids.append(bit_to_token(b)); bit_sigs.append(i); step_types.append(STEP_INPUT)
    for i, b in enumerate(y_bits):
        token_ids.append(bit_to_token(b)); bit_sigs.append(i); step_types.append(STEP_INPUT)

    # log: list of dicts with token, step, source, gt_token (if known)
    log = []
    for t in token_ids:
        log.append({'token': t, 'step': 'INPUT', 'source': 'input'})

    # Ground truth trace for comparison
    gt_seq, _ = generate_single_level_trace(x, y, n_bits, base_case_bits)
    gt_tokens = [t for t, _ in gt_seq]
    gt_steps = []
    for t, p in gt_seq:
        if t in ALL_TAG_IDS:
            gt_steps.append(TAG_NAMES.get(t, '?'))
        else:
            gt_steps.append(gt_steps[-1] if gt_steps else '?')

    max_gen = 300; current_step_type = STEP_INPUT; current_bit_idx = 0
    gt_idx = len(token_ids)  # next GT position to compare against
    first_error = None

    for gen_step in range(max_gen):
        next_tok = model_predict_next(model, token_ids, bit_sigs, step_types, n_loops)

        # Compare with ground truth
        gt_tok = gt_tokens[gt_idx] if gt_idx < len(gt_tokens) else None
        gt_step = gt_steps[gt_idx] if gt_idx < len(gt_steps) else '?'

        if next_tok == PAUSE_ID:
            # Check GT expects PAUSE here too
            if first_error is None and gt_tok is not None and gt_tok != PAUSE_ID:
                first_error = {
                    'pos': gt_idx, 'gen_step': gen_step,
                    'predicted': tok_name(PAUSE_ID), 'expected': tok_name(gt_tok),
                    'gt_step': gt_step, 'type': 'tag_error'
                }

            sub = extract_sub_problem(token_ids)
            if sub is None:
                log.append({'token': 'PAUSE_FAIL', 'step': 'ERROR', 'source': 'model'})
                return None, log, first_error or {'pos': gt_idx, 'type': 'pause_extract_fail'}
            a, b, sub_n_bits = sub
            padded = 1
            while padded < sub_n_bits: padded <<= 1
            padded = max(padded, 4)
            sub_result, sub_log, sub_err = diagnostic_inference(
                model, a, b, padded, base_case_bits, n_loops, depth+1)
            if sub_result is None:
                return None, log, first_error or {'pos': gt_idx, 'type': 'recurse_fail'}

            token_ids.append(PAUSE_ID); bit_sigs.append(200+STEP_PAUSE); step_types.append(STEP_PAUSE)
            log.append({'token': PAUSE_ID, 'step': 'PAUSE', 'source': 'model', 'gt': gt_tok})
            gt_idx += 1  # skip PAUSE in GT

            result_n_bits = 2 * sub_n_bits
            result_bits = int_to_bits(sub_result, result_n_bits)
            token_ids.append(RESUME_ID); bit_sigs.append(200+STEP_RESUME); step_types.append(STEP_RESUME)
            log.append({'token': RESUME_ID, 'step': 'RESUME', 'source': 'oracle'})
            gt_idx += 1  # skip RESUME in GT
            for i, bb in enumerate(result_bits):
                t = bit_to_token(bb)
                token_ids.append(t); bit_sigs.append(i); step_types.append(STEP_RESUME)
                log.append({'token': t, 'step': 'RESUME', 'source': 'oracle'})
                gt_idx += 1  # skip RESUME bits in GT
            current_step_type = STEP_RESUME; current_bit_idx = 0
            continue

        elif next_tok == OUTPUT_ID:
            if first_error is None and gt_tok is not None and gt_tok != OUTPUT_ID:
                first_error = {
                    'pos': gt_idx, 'gen_step': gen_step,
                    'predicted': tok_name(OUTPUT_ID), 'expected': tok_name(gt_tok),
                    'gt_step': gt_step, 'type': 'tag_error'
                }
            token_ids.append(OUTPUT_ID); bit_sigs.append(200+STEP_OUTPUT); step_types.append(STEP_OUTPUT)
            log.append({'token': OUTPUT_ID, 'step': 'OUTPUT', 'source': 'model', 'gt': gt_tok})
            gt_idx += 1

            product_n_bits = 2 * n_bits
            output_bits = []
            for i in range(product_n_bits):
                nb = model_predict_next(model, token_ids, bit_sigs, step_types, n_loops)
                gt_out = gt_tokens[gt_idx] if gt_idx < len(gt_tokens) else None
                if first_error is None and gt_out is not None and nb != gt_out:
                    first_error = {
                        'pos': gt_idx, 'gen_step': gen_step + 1 + i,
                        'predicted': tok_name(nb), 'expected': tok_name(gt_out),
                        'gt_step': 'OUTPUT', 'type': 'bit_error'
                    }
                token_ids.append(nb); bit_sigs.append(i); step_types.append(STEP_OUTPUT)
                output_bits.append(token_to_bit(nb))
                log.append({'token': nb, 'step': 'OUTPUT', 'source': 'model', 'gt': gt_out})
                gt_idx += 1
            return bits_to_int(output_bits), log, first_error

        else:
            # Normal model-generated token
            if first_error is None and gt_tok is not None and next_tok != gt_tok:
                is_bit = next_tok in (BIT_0_ID, BIT_1_ID) and gt_tok in (BIT_0_ID, BIT_1_ID)
                first_error = {
                    'pos': gt_idx, 'gen_step': gen_step,
                    'predicted': tok_name(next_tok), 'expected': tok_name(gt_tok),
                    'gt_step': gt_step, 'type': 'bit_error' if is_bit else 'tag_error'
                }

            if next_tok in ALL_TAG_IDS:
                current_step_type = TAG_TO_STEP.get(next_tok, current_step_type)
                current_bit_idx = 0; bit_sigs.append(200 + current_step_type)
            else:
                bit_sigs.append(current_bit_idx); current_bit_idx += 1
            token_ids.append(next_tok); step_types.append(current_step_type)
            log.append({'token': next_tok, 'step': TAG_NAMES.get(next_tok, '?'), 'source': 'model', 'gt': gt_tok})
            gt_idx += 1

    return None, log, first_error or {'pos': -1, 'type': 'timeout'}


# ===== RUN DIAGNOSIS =====
print('Running 8-bit autoregressive diagnosis (100 examples)...')
t0 = time.time()
rng_diag = pyrandom.Random(999)
n_samples = 100

errors_by_step = {}
errors_by_type = {}
first_error_positions = []
total_correct = 0
error_details = []

for trial in range(n_samples):
    a = rng_diag.randint(0, 255); b = rng_diag.randint(0, 255)
    expected = a * b
    result, trace_log, first_err = diagnostic_inference(model, a, b, 8)

    if result == expected:
        total_correct += 1
        continue

    step = first_err.get('gt_step', first_err.get('type', 'unknown'))
    etype = first_err.get('type', 'unknown')
    errors_by_step[step] = errors_by_step.get(step, 0) + 1
    errors_by_type[etype] = errors_by_type.get(etype, 0) + 1
    if first_err.get('pos') is not None and first_err['pos'] >= 0:
        first_error_positions.append(first_err['pos'])

    if len(error_details) < 15:
        error_details.append({
            'problem': f'{a}*{b}={expected}',
            'got': result,
            'first_err': first_err,
        })

elapsed = time.time() - t0
n_errors = n_samples - total_correct

print(f'\n{"=" * 65}')
print(f'PHASE 7 DIAGNOSIS: {total_correct}/{n_samples} correct ({total_correct/n_samples:.1%})  [{elapsed:.0f}s]')
print(f'{"=" * 65}')

print(f'\n--- First error by STEP TYPE (where does it go wrong?) ---')
for step, count in sorted(errors_by_step.items(), key=lambda x: -x[1]):
    pct = count / max(n_errors, 1) * 100
    bar = '#' * int(pct / 2)
    print(f'  {step:<15} {count:3d} ({pct:4.0f}%) {bar}')

print(f'\n--- Error TYPE (bit flip vs structural) ---')
for etype, count in sorted(errors_by_type.items(), key=lambda x: -x[1]):
    pct = count / max(n_errors, 1) * 100
    print(f'  {etype:<20} {count:3d} ({pct:4.0f}%)')

if first_error_positions:
    positions = np.array(first_error_positions)
    print(f'\n--- First error POSITION in trace ---')
    print(f'  Mean: {positions.mean():.1f}, Median: {np.median(positions):.0f}, '
          f'Min: {positions.min()}, Max: {positions.max()}')

print(f'\n--- First 15 error details ---')
for d in error_details:
    err = d['first_err']
    print(f"  {d['problem']:>20} got {str(d['got']):>8} | "
          f"pos {err.get('pos','?'):>3} step={err.get('gt_step','?'):<10} "
          f"{err.get('predicted','?')} vs {err.get('expected','?')} ({err.get('type','?')})")

print(f'\n>>> STOP HERE. Read the output above before running cell 6. <<<')


In [ ]:
# Cell 6: Constrained Decoding — restrict model to valid next tokens
#
# The Karatsuba trace has a strict grammar. At each point we know which
# tokens are valid. This cell implements constrained decoding that masks
# out impossible tokens before argmax.

# --- Grammar: what tokens can follow what? ---
# After INPUT tag: n_bits*2 bits then a tag (SPLIT or BASE_MUL)
# After SPLIT tag: bits then M0
# After M0/M2/M1: bits then PAUSE
# After RESUME: bits then next tag
# After ADD: bits then M1
# After SUB: bits then COMBINE
# After COMBINE: bits then OUTPUT
# After OUTPUT: product bits then done
# After BASE_MUL: bits then OUTPUT

# Expected tag sequence for 8-bit recursive case:
# INPUT -> SPLIT -> M0 -> PAUSE/RESUME -> M2 -> PAUSE/RESUME -> ADD -> M1 -> PAUSE/RESUME -> SUB -> COMBINE -> OUTPUT
RECURSIVE_TAG_ORDER = [INPUT_ID, SPLIT_ID, SUB_MUL_0_ID, PAUSE_ID,  # + RESUME injected
                       SUB_MUL_2_ID, PAUSE_ID,
                       ADD_ID, SUB_MUL_1_ID, PAUSE_ID,
                       SUB_ID, COMBINE_ID, OUTPUT_ID]

# Expected bit counts between tags for 8-bit (n_bits=8, half=4, hi=4)
def get_expected_bit_counts(n_bits):
    """Return list of (tag, n_bits_after_tag) for each tag in the trace."""
    half = n_bits // 2; hi = n_bits - half
    sum_bits = max(half, hi) + 1
    product_bits = 2 * n_bits
    return {
        'INPUT': n_bits * 2,       # x_bits + y_bits
        'SPLIT': hi + half + hi + half,  # x_hi, x_lo, y_hi, y_lo
        'M0': half * 2,            # x_lo, y_lo operands
        'M2': hi * 2,             # x_hi, y_hi operands
        'ADD': sum_bits * 2,       # sum_x, sum_y
        'M1': sum_bits * 2,        # sum_x, sum_y operands
        'SUB': product_bits,       # z1 result
        'COMBINE': product_bits,   # full result
        'OUTPUT': product_bits,    # final answer
        'BASE_MUL': product_bits,  # base case product
    }


def constrained_predict(model, token_ids, bit_sigs, step_types,
                         valid_tokens, n_loops=8):
    """Predict next token, but only from valid_tokens set."""
    _, logits = model_predict_next_with_logits(
        model, token_ids, bit_sigs, step_types, n_loops)
    # Mask out invalid tokens
    mask = jnp.full(VOCAB_SIZE, -1e9)
    for v in valid_tokens:
        mask = mask.at[v].set(0.0)
    masked_logits = logits + mask
    return int(jnp.argmax(masked_logits))


def recursive_inference_constrained(model, x, y, n_bits, base_case_bits=4, n_loops=8, depth=0):
    """Autoregressive inference with grammar-constrained decoding."""
    x_bits = int_to_bits(x, n_bits); y_bits = int_to_bits(y, n_bits)
    token_ids = [INPUT_ID]
    bit_sigs = [200 + STEP_INPUT]
    step_types = [STEP_INPUT]
    for i, b in enumerate(x_bits):
        token_ids.append(bit_to_token(b)); bit_sigs.append(i); step_types.append(STEP_INPUT)
    for i, b in enumerate(y_bits):
        token_ids.append(bit_to_token(b)); bit_sigs.append(i); step_types.append(STEP_INPUT)

    total_tokens = len(token_ids)
    current_step_type = STEP_INPUT; current_bit_idx = 0

    is_base_case = n_bits <= base_case_bits
    bit_counts = get_expected_bit_counts(n_bits)
    product_bits = 2 * n_bits

    # State machine: track expected tag sequence
    if is_base_case:
        tag_sequence = [BASE_MUL_ID, OUTPUT_ID]
    else:
        tag_sequence = [SPLIT_ID, SUB_MUL_0_ID, PAUSE_ID,
                        SUB_MUL_2_ID, PAUSE_ID,
                        ADD_ID, SUB_MUL_1_ID, PAUSE_ID,
                        SUB_ID, COMBINE_ID, OUTPUT_ID]

    tag_step_names = {
        SPLIT_ID:'SPLIT', SUB_MUL_0_ID:'M0', SUB_MUL_2_ID:'M2',
        ADD_ID:'ADD', SUB_MUL_1_ID:'M1', SUB_ID:'SUB',
        COMBINE_ID:'COMBINE', OUTPUT_ID:'OUTPUT', BASE_MUL_ID:'BASE_MUL',
    }
    tag_idx = 0  # which tag we expect next
    bits_remaining = bit_counts.get('INPUT', 0) - n_bits * 2  # already consumed input bits
    # After INPUT tag + input bits, we expect the first tag from tag_sequence
    # We've already placed all input bits, so bits_remaining for the input region = 0
    bits_remaining = 0  # input bits already placed
    current_region = 'INPUT'

    max_gen = 300
    for _ in range(max_gen):
        if tag_idx >= len(tag_sequence):
            break  # done

        if bits_remaining > 0:
            # Expecting a bit token
            valid = {BIT_0_ID, BIT_1_ID}
        else:
            # Expecting the next tag
            valid = {tag_sequence[tag_idx]}

        next_tok = constrained_predict(model, token_ids, bit_sigs, step_types, valid, n_loops)

        if next_tok == PAUSE_ID:
            sub = extract_sub_problem(token_ids)
            if sub is None:
                return None, total_tokens
            a, b, sub_n_bits = sub
            padded = 1
            while padded < sub_n_bits: padded <<= 1
            padded = max(padded, 4)
            sub_result, sub_tok = recursive_inference_constrained(
                model, a, b, padded, base_case_bits, n_loops, depth+1)
            total_tokens += sub_tok
            if sub_result is None:
                return None, total_tokens

            token_ids.append(PAUSE_ID); bit_sigs.append(200+STEP_PAUSE); step_types.append(STEP_PAUSE)
            result_n_bits = 2 * sub_n_bits
            result_bits = int_to_bits(sub_result, result_n_bits)
            token_ids.append(RESUME_ID); bit_sigs.append(200+STEP_RESUME); step_types.append(STEP_RESUME)
            for i, bb in enumerate(result_bits):
                token_ids.append(bit_to_token(bb)); bit_sigs.append(i); step_types.append(STEP_RESUME)
            current_step_type = STEP_RESUME; current_bit_idx = 0
            tag_idx += 1  # consumed PAUSE, move to next expected tag
            # Next region: set bits_remaining for the tag after PAUSE
            if tag_idx < len(tag_sequence):
                next_tag = tag_sequence[tag_idx]
                # The next tag comes immediately (no bits between RESUME and next tag)
                bits_remaining = 0
            continue

        elif next_tok == OUTPUT_ID:
            token_ids.append(OUTPUT_ID); bit_sigs.append(200+STEP_OUTPUT); step_types.append(STEP_OUTPUT)
            tag_idx += 1
            # Generate exactly product_bits output bits (constrained to BIT_0/BIT_1)
            output_bits = []
            for i in range(product_bits):
                nb = constrained_predict(model, token_ids, bit_sigs, step_types,
                                         {BIT_0_ID, BIT_1_ID}, n_loops)
                token_ids.append(nb); bit_sigs.append(i); step_types.append(STEP_OUTPUT)
                output_bits.append(token_to_bit(nb))
            total_tokens += len(token_ids)
            return bits_to_int(output_bits), total_tokens

        elif next_tok in ALL_TAG_IDS:
            # A structural tag
            current_step_type = TAG_TO_STEP.get(next_tok, current_step_type)
            current_bit_idx = 0
            token_ids.append(next_tok); bit_sigs.append(200 + current_step_type)
            step_types.append(current_step_type)
            tag_idx += 1
            # Set bits_remaining for this region
            region_name = tag_step_names.get(next_tok, '?')
            bits_remaining = bit_counts.get(region_name, 0)

        else:
            # Bit token
            bit_sigs.append(current_bit_idx); current_bit_idx += 1
            token_ids.append(next_tok); step_types.append(current_step_type)
            bits_remaining -= 1

    return None, total_tokens

print('Constrained decoding defined. Run cell 7 to evaluate.')


In [ ]:
# Cell 7: Evaluate constrained decoding vs unconstrained
import time

# --- Unconstrained baseline (Phase 7 original) ---
def recursive_inference_unconstrained(model, x, y, n_bits, base_case_bits=4, n_loops=8, depth=0):
    """Original Phase 7 inference (no constraints)."""
    x_bits = int_to_bits(x, n_bits); y_bits = int_to_bits(y, n_bits)
    token_ids = [INPUT_ID]
    bit_sigs = [200 + STEP_INPUT]
    step_types = [STEP_INPUT]
    for i, b in enumerate(x_bits):
        token_ids.append(bit_to_token(b)); bit_sigs.append(i); step_types.append(STEP_INPUT)
    for i, b in enumerate(y_bits):
        token_ids.append(bit_to_token(b)); bit_sigs.append(i); step_types.append(STEP_INPUT)
    total_tokens = len(token_ids)
    max_gen = 300; current_step_type = STEP_INPUT; current_bit_idx = 0
    for _ in range(max_gen):
        next_tok = model_predict_next(model, token_ids, bit_sigs, step_types, n_loops)
        if next_tok == PAUSE_ID:
            sub = extract_sub_problem(token_ids)
            if sub is None: return None, total_tokens
            a, b, sub_n_bits = sub
            padded = 1
            while padded < sub_n_bits: padded <<= 1
            padded = max(padded, 4)
            sub_result, sub_tok = recursive_inference_unconstrained(
                model, a, b, padded, base_case_bits, n_loops, depth+1)
            total_tokens += sub_tok
            if sub_result is None: return None, total_tokens
            token_ids.append(PAUSE_ID); bit_sigs.append(200+STEP_PAUSE); step_types.append(STEP_PAUSE)
            result_n_bits = 2 * sub_n_bits
            result_bits = int_to_bits(sub_result, result_n_bits)
            token_ids.append(RESUME_ID); bit_sigs.append(200+STEP_RESUME); step_types.append(STEP_RESUME)
            for i, bb in enumerate(result_bits):
                token_ids.append(bit_to_token(bb)); bit_sigs.append(i); step_types.append(STEP_RESUME)
            current_step_type = STEP_RESUME; current_bit_idx = 0
            continue
        elif next_tok == OUTPUT_ID:
            token_ids.append(OUTPUT_ID); bit_sigs.append(200+STEP_OUTPUT); step_types.append(STEP_OUTPUT)
            product_n_bits = 2 * n_bits
            output_bits = []
            for i in range(product_n_bits):
                nb = model_predict_next(model, token_ids, bit_sigs, step_types, n_loops)
                token_ids.append(nb); bit_sigs.append(i); step_types.append(STEP_OUTPUT)
                output_bits.append(token_to_bit(nb))
            total_tokens += len(token_ids)
            return bits_to_int(output_bits), total_tokens
        else:
            if next_tok in ALL_TAG_IDS:
                current_step_type = TAG_TO_STEP.get(next_tok, current_step_type)
                current_bit_idx = 0; bit_sigs.append(200 + current_step_type)
            else:
                bit_sigs.append(current_bit_idx); current_bit_idx += 1
            token_ids.append(next_tok); step_types.append(current_step_type)
    return None, total_tokens


# --- Run both ---
print('=' * 65)
print('CONSTRAINED vs UNCONSTRAINED DECODING')
print('=' * 65)

for label, infer_fn in [('Unconstrained', recursive_inference_unconstrained),
                         ('Constrained', recursive_inference_constrained)]:
    for n_bits in [4, 8]:
        max_val = 1 << n_bits
        n_test = 100 if n_bits <= 8 else 50
        rng_eval = pyrandom.Random(999)
        correct, total, errors = 0, 0, []
        t0 = time.time()
        for _ in range(n_test):
            a = rng_eval.randint(0, max_val - 1); b = rng_eval.randint(0, max_val - 1)
            result, _ = infer_fn(model, a, b, n_bits)
            if result == a * b: correct += 1
            else: errors.append((a, b, a*b, result))
            total += 1
        elapsed = time.time() - t0
        print(f'  {label:15s} {n_bits:2d}-bit: {correct}/{total} = {correct/total:.1%}  ({elapsed:.0f}s)')
        if errors[:2]:
            for a, b, exp, got in errors[:2]:
                print(f'    Error: {a}*{b}={exp}, got {got}')

print()
print('If constrained >> unconstrained: errors were structural (tags).')
print('If constrained ~= unconstrained: errors are in bit computation.')
print()
print('>>> Based on results, decide whether to proceed to cell 8 (DAgger). <<<')


In [ ]:
# Cell 8: DAgger WITHOUT BACKTRACK — scheduled sampling on model prefixes
#
# Only run this if constrained decoding didn't solve it.
# This trains the model on its own autoregressive prefixes
# with oracle supervision (no BACKTRACK token).
import time

# Training setup
schedule = optax.warmup_cosine_decay_schedule(
    init_value=1e-6, peak_value=5e-4, warmup_steps=200,
    decay_steps=30000, end_value=1e-6
)
optimizer = optax.adamw(learning_rate=schedule, weight_decay=0.15)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

def make_train_step(n_loops):
    @eqx.filter_jit
    def train_step(model, opt_state, token_ids, pad_mask, bit_sig, step_type):
        input_ids = token_ids[:, :-1]
        target_ids = token_ids[:, 1:]
        inp_bs = bit_sig[:, :-1]; inp_st = step_type[:, :-1]
        mask = make_pause_resume_mask(token_ids, pad_mask)
        zeros = jnp.zeros_like(inp_bs)
        def loss_fn(model):
            def fwd(ids, bs, st):
                return model(ids, n_loops, bs, zeros[0], zeros[0], st)
            logits = jax.vmap(fwd)(input_ids, inp_bs, inp_st)
            log_probs = jax.nn.log_softmax(logits, axis=-1)
            targets_oh = jax.nn.one_hot(target_ids, VOCAB_SIZE)
            tok_loss = -jnp.sum(targets_oh * log_probs, axis=-1)
            return jnp.sum(tok_loss * mask) / jnp.maximum(jnp.sum(mask), 1.0)
        loss, grads = eqx.filter_value_and_grad(loss_fn)(model)
        updates, new_opt = optimizer.update(grads, opt_state, eqx.filter(model, eqx.is_array))
        return eqx.apply_updates(model, updates), new_opt, loss
    return train_step

train_steps = {n: make_train_step(n) for n in [4, 6, 8]}


def dagger_generate_oracle_trajectory(model, clean_example, rollout_len, n_loops=8):
    """Generate trajectory: teacher-forced prefix, then model rollout,
    but always append the ORACLE (correct) next token.
    No BACKTRACK — just expose model to its own prefixes."""
    token_ids_gt, bit_sigs_gt, step_types_gt = clean_example[:3]
    seq_len = len(token_ids_gt)

    max_start = max(1, seq_len - rollout_len - 1)
    start = pyrandom.randint(1, max_start)

    # Start with teacher-forced prefix
    traj_tokens = list(token_ids_gt[:start])
    traj_bsigs = list(bit_sigs_gt[:start])
    traj_stypes = list(step_types_gt[:start])

    for i in range(rollout_len):
        gt_pos = start + i
        if gt_pos >= seq_len:
            break

        # Model predicts (we don't use the prediction, just expose it to its own prefix)
        # But we ALWAYS append the oracle token
        _ = model_predict_next(model, traj_tokens, traj_bsigs, traj_stypes, n_loops)

        # Append oracle token
        traj_tokens.append(int(token_ids_gt[gt_pos]))
        traj_bsigs.append(int(bit_sigs_gt[gt_pos]))
        traj_stypes.append(int(step_types_gt[gt_pos]))

    return (np.array(traj_tokens, dtype=np.int32),
            np.array(traj_bsigs, dtype=np.int32),
            np.array(traj_stypes, dtype=np.int32))


# --- DAgger loop ---
print('=== DAgger WITHOUT BACKTRACK ===')
print('Training on model prefixes with oracle supervision')
print()

rng = random.Random(42)
clean_8bit = [ex for ex in train_8bit]  # all clean (no backtrack augmentation here)
rollout_schedule = [(0, 1), (25, 2), (50, 4), (75, 8), (100, 16)]
current_rollout = 1
best_tf = 0.0

t0 = time.time()
for epoch in range(200):
    for sched_epoch, sched_len in rollout_schedule:
        if epoch >= sched_epoch:
            current_rollout = sched_len

    epoch_loss, n_batches = 0.0, 0

    # Teacher-forced on 4-bit (anchor)
    n_loops = rng.choice([4, 6, 8])
    step_fn = train_steps[n_loops]
    for batch in get_batches(train_4bit, batch_size=64, max_len=ml4):
        tk=jnp.array(batch['token_ids']); pm=jnp.array(batch['pad_mask'])
        bs=jnp.array(batch['bit_sig']); st=jnp.array(batch['step_type'])
        model, opt_state, loss = step_fn(model, opt_state, tk, pm, bs, st)
        epoch_loss += float(loss); n_batches += 1

    # Teacher-forced on CLEAN 8-bit (strong anchor — key difference from Phase 8)
    n_loops = rng.choice([4, 6, 8])
    step_fn = train_steps[n_loops]
    for batch in get_batches(train_8bit, batch_size=16, max_len=ml8):
        tk=jnp.array(batch['token_ids']); pm=jnp.array(batch['pad_mask'])
        bs=jnp.array(batch['bit_sig']); st=jnp.array(batch['step_type'])
        model, opt_state, loss = step_fn(model, opt_state, tk, pm, bs, st)
        epoch_loss += float(loss); n_batches += 1

    # DAgger: small batch of model-prefix trajectories (trained as normal sequences)
    if current_rollout > 0:
        sample_size = min(32, len(clean_8bit))
        sampled = pyrandom.sample(clean_8bit, sample_size)
        # Build DAgger batch as padded arrays
        dagger_max_len = ml8 + current_rollout * 2
        d_tk = np.zeros((sample_size, dagger_max_len), dtype=np.int32)
        d_bs = np.zeros((sample_size, dagger_max_len), dtype=np.int32)
        d_st = np.zeros((sample_size, dagger_max_len), dtype=np.int32)
        d_pm = np.zeros((sample_size, dagger_max_len), dtype=np.float32)
        for i, ex in enumerate(sampled):
            traj = dagger_generate_oracle_trajectory(model, ex, current_rollout)
            sl = len(traj[0])
            d_tk[i, :sl] = traj[0]; d_bs[i, :sl] = traj[1]; d_st[i, :sl] = traj[2]
            d_pm[i, :sl] = 1.0
        step_fn = train_steps[8]
        # Train in mini-batches
        for s in range(0, sample_size, 16):
            e = min(s + 16, sample_size)
            model, opt_state, loss = step_fn(
                model, opt_state,
                jnp.array(d_tk[s:e]), jnp.array(d_pm[s:e]),
                jnp.array(d_bs[s:e]), jnp.array(d_st[s:e]))
            epoch_loss += float(loss); n_batches += 1

    if epoch % 25 == 0:
        avg = epoch_loss / max(n_batches, 1)
        c8, t8 = tf_eval(model, eval_8bit, eml8)
        tf_acc = c8/t8
        elapsed = time.time() - t0
        improved = ' *' if tf_acc > best_tf else ''
        best_tf = max(best_tf, tf_acc)
        print(f'Epoch {epoch:4d} | Loss: {avg:.4f} | 8-bit TF: {c8}/{t8} = {tf_acc:.1%} | '
              f'rollout={current_rollout} | {elapsed:.0f}s{improved}')

        # Early stop if TF drops below 90%
        if tf_acc < 0.90 and epoch > 50:
            print(f'\nWARNING: TF accuracy dropped below 90%. Stopping early.')
            break

elapsed = time.time() - t0
print(f'\nDAgger complete in {elapsed:.0f}s.')


In [ ]:
# Cell 9: Final evaluation — 4/8/16/32-bit
print('=' * 65)
print('FINAL EVALUATION — AUTOREGRESSIVE RECURSIVE INFERENCE')
print('=' * 65)

results_unc = {}
results_con = {}

for n_bits in [4, 8, 16, 32]:
    max_val = 1 << n_bits
    n_test = 100 if n_bits <= 16 else 30
    rng_eval = pyrandom.Random(999)

    for label, infer_fn, results_dict in [
        ('Unconstrained', recursive_inference_unconstrained, results_unc),
        ('Constrained', recursive_inference_constrained, results_con),
    ]:
        correct, total = 0, 0
        rng_eval = pyrandom.Random(999)  # reset for fair comparison
        t0 = time.time()
        for _ in range(n_test):
            a = rng_eval.randint(0, max_val - 1); b = rng_eval.randint(0, max_val - 1)
            result, _ = infer_fn(model, a, b, n_bits)
            if result == a * b: correct += 1
            total += 1
        elapsed = time.time() - t0
        acc = correct / total
        results_dict[n_bits] = acc
        print(f'  {label:15s} {n_bits:2d}-bit: {correct}/{total} = {acc:.1%}  ({elapsed:.0f}s)')

print(f'\n{"=" * 65}')
print('COMPARISON TABLE')
print(f'{"=" * 65}')
print(f'{"Method":<42} {"4-bit":>8} {"8-bit":>8} {"16-bit":>8} {"32-bit":>8}')
print('-' * 74)
print(f'{"Phase 5: Flat sequence":<42} {"N/A":>8} {"99%":>8} {"0%":>8} {"N/A":>8}')
print(f'{"Phase 6: Teacher-forced recursive":<42} {"100%":>8} {"100%":>8} {"100%":>8} {"100%":>8}')
print(f'{"Phase 7: Autoregressive (original)":<42} {"N/A":>8} {"66%":>8} {"N/A":>8} {"N/A":>8}')
print(f'{"Phase 8: BACKTRACK + DAgger":<42} {"72%":>8} {"0%":>8} {"0%":>8} {"0%":>8}')
print(f'{"Phase 9: Unconstrained + DAgger":<42}', end='')
for nb in [4, 8, 16, 32]:
    print(f'{results_unc.get(nb, 0):>7.1%} ', end='')
print()
print(f'{"Phase 9: Constrained + DAgger":<42}', end='')
for nb in [4, 8, 16, 32]:
    print(f'{results_con.get(nb, 0):>7.1%} ', end='')
print()
print(f'{"=" * 65}')


In [ ]:
# Cell 10: Save model
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
CKPT_DIR = '/content/drive/MyDrive/karatsuba_checkpoints/'
os.makedirs(CKPT_DIR, exist_ok=True)
eqx.tree_serialise_leaves(os.path.join(CKPT_DIR, 'model_phase9_dagger_constrained.eqx'), model)
print(f'Model saved to {CKPT_DIR}')
